In [1]:
!pip install nvcc4jupyter
%load_ext nvcc4jupyter

Detected platform "Colab". Running its setup...
Source files will be saved in "/tmp/tmpvsp_qszg".


In [3]:
%%cuda
#include <iostream>

__global__ void mapThreads2D(int* matrix, int width, int height) {
    // 1. Identify 2D coordinates
    int col = blockIdx.x * blockDim.x + threadIdx.x;
    int row = blockIdx.y * blockDim.y + threadIdx.y;

    // 2. Map to 1D Linear Index
    if (col < width && row < height) {
        int index = row * width + col;

        // 3. Store the ID to see the mapping pattern
        matrix[index] = index;
    }
}

int main() {
    int width = 6;
    int height = 4;
    size_t size = width * height * sizeof(int);

    int *h_matrix = (int*)malloc(size);
    int *d_matrix;
    cudaMalloc(&d_matrix, size);

    // Launch Config:
    // We use 3x2 threads per block.
    // To cover 6x4, we need 2x2 blocks.
    dim3 threadsPerBlock(3, 2);
    dim3 blocksPerGrid(2, 2);

    mapThreads2D<<<blocksPerGrid, threadsPerBlock>>>(d_matrix, width, height);

    cudaMemcpy(h_matrix, d_matrix, size, cudaMemcpyDeviceToHost);

    std::cout << "Thread Mapping (Global Linear IDs):" << std::endl;
    for(int i = 0; i < height; i++) {
        for(int j = 0; j < width; j++) {
            // Print with padding for alignment
            printf("%2d ", h_matrix[i * width + j]);
        }
        std::cout << std::endl;
    }

    cudaFree(d_matrix);
    free(h_matrix);
    return 0;
}

Thread Mapping (Global Linear IDs):
 0  1  2  3  4  5 
 6  7  8  9 10 11 
12 13 14 15 16 17 
18 19 20 21 22 23 

